In [1]:
%pip install natasha


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import re
import os
import time
import csv
from collections import Counter, defaultdict
from tqdm import tqdm
import pymorphy3
from natasha import Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger, Doc

In [3]:
def simple_tokenize(text):
    return re.compile(r"[A-Za-zА-Яа-яЁё0-9\-']+").findall(text)

def lemmatize_with_pymorphy3(morph, tokens):
    lemmas = []
    for tok in tokens:
        key = tok.lower()
        if not tok.isalpha():
            lemma = "UNK"
        else:
            try:
                parses = morph.parse(key)
                lemma = parses[0].normal_form if parses else "UNK"
            except Exception:
                lemma = "UNK"
        lemmas.append(lemma)
    return lemmas

def lemmatize_with_natasha(segmenter, morph_tagger, morph_vocab, text):
    doc = Doc(text)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)

    tokens = []
    lemmas = []


    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        tok = token.text
        if not tok.isalpha():
            lem = "UNK"
        elif getattr(token, "lemma", None):
            lem = token.lemma.lower()
        else:
            lem = "UNK"
        tokens.append(tok)
        lemmas.append(lem)

    return tokens, lemmas

def save_freq(counter, path):
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['lemma','freq'])
        for lemma, freq in counter.most_common():
            writer.writerow([lemma, freq])

In [4]:
# инициализация анализаторов
morph = pymorphy3.MorphAnalyzer()
segmenter = Segmenter()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)
morph_vocab = MorphVocab()

In [5]:
#pymorphy3
freq_pym = Counter()
pym_token_count = 0
lemmas_pym = []

csv_path = os.path.join("files_hw3", 'tokens_lemmas_pymorphy3.csv')
with open(csv_path, 'w', encoding='utf-8', newline='') as outf:
    writer = csv.writer(outf)
    writer.writerow(['token', 'lemma'])

    time_start = time.time()
    with open("corpus.txt", 'r', encoding='utf-8') as inf:
        for line in inf:
            text = line.rstrip('\n')
            if not text:
                continue

            toks = simple_tokenize(text)
            lemmas = lemmatize_with_pymorphy3(morph, toks)

            lemmas_pym.extend(lemmas)
            freq_pym.update(lemmas)
            pym_token_count += len(toks)

            for tok, lem in zip(toks, lemmas):
                writer.writerow([tok, lem])

t_pym = time.time() - time_start
save_freq(freq_pym, os.path.join("files_hw3", 'freq_pymorphy3.csv'))

print("Time:", t_pym)
print("Total tokens:", pym_token_count)

Time: 0.7050571441650391
Total tokens: 6270


In [6]:
#natasha
freq_nat = Counter()
nat_token_count = 0
lemmas_nat = []

csv_path = os.path.join("files_hw3", 'tokens_lemmas_natasha.csv')
with open(csv_path, 'w', encoding='utf-8', newline='') as outf:
    writer = csv.writer(outf)
    writer.writerow(['token', 'lemma'])

    time_start = time.time()
    with open("corpus.txt", 'r', encoding='utf-8') as inf:
        for line in inf:
            text = line.rstrip('\n')
            if not text:
                continue

            toks_nat, lem_nat = lemmatize_with_natasha(segmenter, morph_tagger, morph_vocab, text)

            lemmas_nat.extend(lem_nat)
            freq_nat.update(lem_nat)
            nat_token_count += len(toks_nat)

            for tok, lem in zip(toks_nat, lem_nat):
                writer.writerow([tok, lem])

t_nat = time.time() - time_start
save_freq(freq_nat, os.path.join("files_hw3", 'freq_natasha.csv'))

print("Time (sec):", t_nat)
print("Total tokens:", nat_token_count)

Time (sec): 1.122849941253662
Total tokens: 7904
